# lecture des datasets du catalogue data.gouv
étape 0 d'un workflow de catalogage des données utiles pour l'analyse des mobilités
alternative simple par rapport à lecture_datasets_api qui interroge l'API (par pages) pour récupérer la structure complètes des infos attachées aux datasets en json : ici on télécharge directement une structure simplifiées des données, le fichier csv des datasets disponibles dans le catalogue data.gouv.fr


In [80]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import csv
import io
import re
import sys

from pathlib import Path

import requests

import pandas as pd

DATAGOUV_DATASET_URL = (
    "https://www.data.gouv.fr/api/1/datasets/catalogue-des-donnees-de-data-gouv-fr/"
)

OUTPUT_COLUMNS = [
    "id", "url", "slug", "tags", "title", "description", "badges", 
    "owner_id", "organization", "organization_id", "created_at",
    "last_modified", "resources_count", "resources_formats",
]



In [75]:
def resolve_catalog_download_url(session: requests.Session) -> str:
    """Interroge l'API data.gouv.fr pour trouver l'URL de téléchargement
    de la dernière ressource CSV du jeu de données 'catalogue des données de data.gouv.fr'."""
    resp = session.get(DATAGOUV_DATASET_URL, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    for resource in data.get("resources", []):
        fmt = (resource.get("format") or "").lower()
        title = (resource.get("title") or "").lower()
        if fmt == "csv" or title.endswith(".csv"):
            return resource["url"]
    raise RuntimeError(
        "Aucune ressource CSV trouvée dans les métadonnées du dataset catalogue data.gouv.fr. "
        "Vérifiez manuellement la page : "
        "https://www.data.gouv.fr/datasets/catalogue-des-donnees-de-data-gouv-fr"
    )


In [76]:
def load_datagouv_catalog_df(local_path: str | None = None, delimiter: str = ";"):
    """Télécharge (ou lit localement) le catalogue CSV data.gouv.fr et retourne un dataframe des datasets de data.gouv.fr avec quelques champs 
    """
    if local_path:
        raw_bytes = Path(local_path).read_bytes()
    else:
        with requests.Session() as session:
            download_url = resolve_catalog_download_url(session)
            print(f"Téléchargement du catalogue depuis : {download_url}")
            resp = session.get(download_url, timeout=120)
            resp.raise_for_status()
            raw_bytes = resp.content

    text = raw_bytes.decode("utf-8", errors="replace")
    source = local_path if local_path else io.StringIO(text) 
    
    df = pd.read_csv(
        source,
        sep=delimiter,
        engine="python",
        on_bad_lines="warn",
        dtype=str,
        keep_default_na=False,
    )
    df=df[df["archived"] == "False"]
    df=df[OUTPUT_COLUMNS]
    print(f"{df.shape} dataframe des datasets non archivés créé via pandas.")
    return df

In [77]:
#local_catalog_path="data"
df = load_datagouv_catalog_df()


Téléchargement du catalogue depuis : https://static.data.gouv.fr/resources/catalogue-des-donnees-de-data-gouv-fr/20261005-055159/export-dataset-20261005-055159.csv
(74633, 14) dataframe des datasets non archivés créé via pandas.


In [79]:
df.to_csv("data/datasets.csv",sep=";")

In [ ]:

def dataset_matches(row: dict, mots_cles: list[str]) -> bool:
    haystack = normalize(
        " ".join([
            get_field(row, "title"),
            get_field(row, "description"),
            get_field(row, "tags"),
        ])
    )
    return any(normalize(mc) in haystack for mc in mots_cles)


In [ ]:

def normalize(text: str) -> str:
    """Minuscule + suppression des accents, pour une comparaison robuste."""
    if text is None:
        return ""
    text = unicodedata.normalize("NFKD", str(text))
    text = "".join(c for c in text if not unicodedata.combining(c))
    return text.lower()


def slugify(name: str) -> str:
    name = normalize(name)
    name = re.sub(r"[^a-z0-9]+", "_", name).strip("_")
    return name or "sans_nom"

def build_output_row(row: dict) -> dict:
    return {
        "id": get_field(row, "id"),
        "url": get_field(row, "url", "page"),
        "slug": get_field(row, "slug"),
        "tags": get_field(row, "tags"),
        "title": get_field(row, "title"),
        "badges": get_field(row, "badges"),
        "owner_id": get_field(row, "owner_id", "owner.id"),
        "organization": get_field(row, "organization", "organization.name"),
        "organization_id": get_field(row, "organization_id", "organization.id"),
        "created_at": get_field(row, "created_at"),
        "last_modified": get_field(row, "last_modified", "last_update"),
        "description_short": get_field(row, "description_short", "description")[:280],
        "resources_count": get_field(row, "resources_count", "nb_resources"),
        "resources_formats": get_field(row, "resources_formats", "resources.format"),
    }
